In [17]:
import os
import json
import pandas as pd
import re
# show all rows
pd.set_option('display.max_rows', None)
# show all columns
pd.set_option('display.max_columns', None)
# show whole value
pd.set_option('display.width', None)

import google_auth_oauthlib.flow
import googleapiclient.discovery
from googleapiclient.errors import HttpError
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials

### List out items on a youtube playlist, given
- client_secret.json that details the oauth credentials
- playlist id

In [22]:
def getYoutubeClient():
    # Disable OAuthlib's HTTPS verification when running locally.
    # *DO NOT* leave this option enabled in production.
    os.environ["OAUTHLIB_INSECURE_TRANSPORT"] = "1"

    api_service_name = "youtube"
    api_version = "v3"
    client_secrets_file = "client_secret.json"
    # scopes = ["https://www.googleapis.com/auth/youtube.readonly"] # only supports reading
    scopes = ["https://www.googleapis.com/auth/youtube"] # supports modifying playlist
    token_file = "token.json"

    credentials = None
    
    # checks for existing credentials from previous run
    if os.path.exists(token_file):
        credentials = Credentials.from_authorized_user_file(token_file, scopes)

    # if no valid credentials are available, get new credentials and create an API client
    if not credentials or not credentials.valid:
        if credentials and credentials.expired and credentials.refresh_token:
            credentials.refresh(Request())
        else: # else, get credentials and create an API client
            flow = google_auth_oauthlib.flow.InstalledAppFlow.from_client_secrets_file(
                client_secrets_file, scopes)
            credentials = flow.run_local_server(port=0)

        # save credentials for next run
        with open(token_file, "w") as token:
            token.write(credentials.to_json())

    return googleapiclient.discovery.build(api_service_name, api_version, credentials=credentials)

def getSongNames(playlist_id):
    # gets credentials/tokenized info to be able to run YouTube API requests
    youtube = getYoutubeClient()

    # holds song data before adding to dataframe
    song_data = []

    # holds the next page token for paginated results, if any
    next_page_token = None

    try: 
        while True:
            # lists out the items in the playlist, with a max of x results
            request = youtube.playlistItems().list(
                part="snippet,contentDetails",
                maxResults=50, # youtube caps this at 50?
                playlistId=playlist_id,
                pageToken=next_page_token # pass token for next batch
            )
            response = request.execute()

            # extract song/item name from the nested maps
            # 'items' key is list of maps -> each map has 'snippet' key -> has 'title' key, which is the song name as the value
            items = response.get("items", [])

            # iterate through the items maps to extract each song
            for item in items:
                snippet = item.get("snippet", {})
                video_id = snippet.get("resourceId", {}).get("videoId", "")
                id = item["id"]
                title = snippet.get("title", "")
                song_data.append((title, id, video_id))

            # check if there is a next page token, if not, break the loop
            next_page_token = response.get("nextPageToken")

            if not next_page_token:
                break
        # return a DataFrame from the song data
        return pd.DataFrame(song_data, columns=["Song_Name", "ID", "Video_ID"])
    except HttpError as e:
        print(f"An HTTP error {e.resp.status} occurred:\n{e.content}")
        return None

def remove_duplicate_playlist_items(youtube, playlist_id):
    seen_video_ids = set()
    items_to_delete = []
    next_page_token = None

    print("Scanning playlist for duplicates...")

    try:
        # Loop through all pages of the playlist
        while True:
            request = youtube.playlistItems().list(
                part="snippet,contentDetails",
                playlistId=playlist_id,
                maxResults=50,
                pageToken=next_page_token
            )
            response = request.execute()
            
            items = response.get("items", [])
            
            for item in items:
                playlist_item_id = item["id"]
                video_id = item["contentDetails"]["videoId"]
                
                if video_id in seen_video_ids:
                    # Duplicate found! Mark this specific playlist entry for deletion
                    items_to_delete.append(playlist_item_id)
                else:
                    seen_video_ids.add(video_id)
            
            # Check if there is another page of videos
            next_page_token = response.get("nextPageToken")
            if not next_page_token:
                break        
    except HttpError as e:
        print(f"An error occurred while scanning the playlist: {e}")
        return

    print(f"Found {len(items_to_delete)} duplicate entries to remove from YouTube.")

    # Delete each duplicate entry from the live playlist
    for playlist_item_id in items_to_delete:
        try:
            youtube.playlistItems().delete(id=playlist_item_id).execute()
            print(f"Successfully deleted duplicate playlist item ID: {playlist_item_id}")
        except HttpError as e:
            print(f"An error occurred while deleting {playlist_item_id}: {e}")


def add_videos_to_playlist(youtube, playlist_id, video_ids):
    success_count = 0
    failed_count = 0

    for video_id in video_ids:
        try:
            request = youtube.playlistItems().insert(
                part="snippet",
                body={
                    "snippet": {
                        "playlistId": playlist_id,
                        "resourceId": {
                            "kind": "youtube#video",
                            "videoId": video_id
                        }
                    }
                }
            )
            response = request.execute()
            print(f"Successfully added: {video_id}")
            success_count += 1
            
        except HttpError as e:
            print(f"Failed to add video {video_id}. Error: {e}")
            failed_count += 1

    print("\n--- Batch Addition Complete ---")
    print(f"Successfully added: {success_count}")
    print(f"Failed: {failed_count}")
    
    return {"success": success_count, "failed": failed_count}

def search_videos_on_playlist(youtube, playlist_id, search_query):
    matching_videos = []
    next_page_token = None

    try:
        while True:
            request = youtube.playlistItems().list(
                part="snippet",
                playlistId=playlist_id,
                maxResults=50,
                pageToken=next_page_token
            )
            response = request.execute()

            items = response.get("items", [])
            
            for item in items:
                title = item["snippet"]["title"]
                video_id = item["snippet"]["resourceId"]["videoId"]

                if re.search(r"\b" + re.escape(search_query) + r"\b", title, re.IGNORECASE):
                    matching_videos.append((title, video_id))

            next_page_token = response.get("nextPageToken")
            if not next_page_token:
                break

        if not matching_videos:
            print(f"No videos found in the playlist matching the query: '{search_query}'")
        else:
            print(f"Found {len(matching_videos)} matching videos for the query: '{search_query}'")

        return pd.DataFrame(matching_videos, columns=["Song_Name", "Video_ID"])
    
    except HttpError as e:
        print(f"An error occurred while searching the playlist: {e}")
        return None

### Get song list from main playlist
- Purpose: check for duplicate songs that have been splintered across many playlists

In [3]:
if __name__ == "__main__":
    # df_1, main playlist
    mainPlaylistId = "PL2vR6i7V0rIqP9CpPl3v2i45vWjPf8zty" # 2021 playlist merge
    df_1 = getSongNames(mainPlaylistId)

    # df_2, list of other playlists
    otherPlaylistIds = [
        "PL2vR6i7V0rIoXdJ7HZT67afj4T24Ji0Od", # 11/25/18 playlist
        "PL2vR6i7V0rIq7-8q76DVIb0LpaD6WGaYI",  # 4/XX/18 Flavor
        "PL2vR6i7V0rIoUaRfs_W6ah_njpnQVmMIJ", # 5/12/19 Chill Throwback
        "PL2vR6i7V0rIowPSTkqE8z89BvlP5lNS4U", # Flavor of the Month
        "PL8D302EA61B45C76E", # Music to get
        "PL2vR6i7V0rIrHq8pqT75Ogl5hT1H3dAuT", # Music to get 2
        "PL2vR6i7V0rIqUoQpsphohyCuj7JJFZvbA"   # Music to get 3
    ]

    # store resulting dfs in a list, then concatenate them into one df
    other_playlist_data = []

    for playlistId in otherPlaylistIds:
        result_df = getSongNames(playlistId)
        other_playlist_data.append(result_df)

    # concatenate the list of other dfs into one df
    df_2 = pd.concat(other_playlist_data, ignore_index=True)

In [4]:
# list of other people's playlists? perhaps can mash it since its not private
# streamersPlaylistIds = [

# ]

# 1242 expected
print(len(df_1))
display(df_1.head(1))

# 67+93+58+224+223+223+375=1263
print(len(df_2))
display(df_2.head(1))

851


,Song_Name,ID,Video_ID
0,Lil Nas X - Old Town Road (Music Video) ft. Bi...,UEwydlI2aTdWMHJJcVA5Q3BQbDN2Mmk0NXZXalBmOHp0eS...,gUcisIlT7sM


1263


,Song_Name,ID,Video_ID
0,Foster The People - Sit Next to Me (Audio),UEwydlI2aTdWMHJJb1hkSjdIWlQ2N2FmajRUMjRKaTBPZC...,BKLVpDTZOPQ


### Check if main_df has any dups in itself

In [5]:
# Keeps only the first occurrence of each unique video name
unique_df_1 = df_1.drop_duplicates(subset=["Song_Name"])

print(f"{len(df_1)} non-unique video names in df_1")
print(f"{len(unique_df_1)} unique video names in unique_df_1")

851 non-unique video names in df_1
762 unique video names in unique_df_1


### Attempt to delete duplicate songs from the main df/playlist

In [6]:
# have to initialize client
# deleting 200 songs maxes out on the daily quota lol
# have this commented out till i need to delete again

youtube_client = getYoutubeClient()

remove_duplicate_playlist_items(youtube_client, mainPlaylistId)

Scanning playlist for duplicates...
Found 67 duplicate entries to remove from YouTube.
Successfully deleted duplicate playlist item ID: UEwydlI2aTdWMHJJcVA5Q3BQbDN2Mmk0NXZXalBmOHp0eS40RENGMkVDNTUyQzRDM0JC
Successfully deleted duplicate playlist item ID: UEwydlI2aTdWMHJJcVA5Q3BQbDN2Mmk0NXZXalBmOHp0eS5CQkIyMUJFOUM2QzJERTlC
Successfully deleted duplicate playlist item ID: UEwydlI2aTdWMHJJcVA5Q3BQbDN2Mmk0NXZXalBmOHp0eS5DMUM4MDIwOUM1QjAxMjEx
Successfully deleted duplicate playlist item ID: UEwydlI2aTdWMHJJcVA5Q3BQbDN2Mmk0NXZXalBmOHp0eS4xQTNBM0VERjQ0QTIzRUMz
Successfully deleted duplicate playlist item ID: UEwydlI2aTdWMHJJcVA5Q3BQbDN2Mmk0NXZXalBmOHp0eS5FRDg3NTZEODVFNDA4MjNE
Successfully deleted duplicate playlist item ID: UEwydlI2aTdWMHJJcVA5Q3BQbDN2Mmk0NXZXalBmOHp0eS5GNDMwNzAzQjc1RTE3N0RE
Successfully deleted duplicate playlist item ID: UEwydlI2aTdWMHJJcVA5Q3BQbDN2Mmk0NXZXalBmOHp0eS5CNzFFRjM0RTVGRDE4MDQ5
Successfully deleted duplicate playlist item ID: UEwydlI2aTdWMHJJcVA5Q3BQbDN2Mmk0NXZXal

In [7]:
# stats post-delete

df_1_no_dups = getSongNames(mainPlaylistId)
print(len(df_1_no_dups))
display(df_1_no_dups.head(1))

784


,Song_Name,ID,Video_ID
0,Lil Nas X - Old Town Road (Music Video) ft. Bi...,UEwydlI2aTdWMHJJcVA5Q3BQbDN2Mmk0NXZXalBmOHp0eS...,gUcisIlT7sM


### Only keep non-duplicate songs within the merged df_2 before comparison

In [8]:
# Keeps only the first occurrence of each unique video name
unique_df_2 = df_2.drop_duplicates(subset=["Song_Name"])

print(f"{len(df_2)} non-unique video names in df_2")
print(f"{len(unique_df_2)} unique video names in unique_df_2")

1263 non-unique video names in df_2
747 unique video names in unique_df_2


### Check how many songs are identical across the main df and merged df

In [10]:
print(f"Main Playlist: {len(df_1_no_dups)}")
print(f"Other Playlists: {len(unique_df_2)}")

Main Playlist: 784
Other Playlists: 747


In [ ]:
# Check how many song names in unique_df_2 are present in df_1_no_dups
matching_count = unique_df_2["Song_Name"].isin(df_1_no_dups["Song_Name"]).sum()

print(f"Number of matching songs: {matching_count}")

Number of matching songs: 731


In [11]:
# Filter unique_df_2 to only keep rows where song_name is found in df_1_no_dups
non_overlapping_songs_df = unique_df_2[~unique_df_2["Song_Name"].isin(df_1_no_dups["Song_Name"])]

print(f"Number of non-overlapping songs: {len(non_overlapping_songs_df)}")

Number of non-overlapping songs: 16


### Take the non over-lapping songs and add them to the main playlist

In [13]:
display(non_overlapping_songs_df.head(1))

,Song_Name,ID,Video_ID
55,Vanessa Hudgens - Say OK (Official Video),UEwydlI2aTdWMHJJb1hkSjdIWlQ2N2FmajRUMjRKaTBPZC...,F5VvvVxuKko


In [15]:
to_add_video_ids_list = non_overlapping_songs_df["Video_ID"].tolist()
# mainPlaylistId = "PL2vR6i7V0rIqP9CpPl3v2i45vWjPf8zty" # 2021 playlist merge

In [16]:
add_videos_to_playlist(youtube_client, mainPlaylistId, to_add_video_ids_list)

Successfully added: F5VvvVxuKko
Successfully added: QvesVAFx8rE
Successfully added: AE3yia1AJeQ
Successfully added: CgiX53hjAPc
Successfully added: 02JMBhxufJw
Successfully added: td-eoeLRYcQ
Successfully added: kWBE0sQC5L8
Successfully added: a81eP2E8MEQ
Successfully added: sU_ByeHJtw8
Successfully added: IsHBnS3ZLSY
Successfully added: Vk5-c_v4gMU
Successfully added: mjli3hj0ZkM
Successfully added: bBJ9RnbK8G4
Successfully added: rO5UDrIoTTs
Successfully added: iPNA9baCj2I
Successfully added: WXKrmua1sY8

--- Batch Addition Complete ---
Successfully added: 16
Failed: 0


{'success': 16, 'failed': 0}

### Search if a song exists on a playlist already

In [23]:
search_query = "Old Town Road"

In [24]:
does_song_exist_df = search_videos_on_playlist(youtube_client, mainPlaylistId, search_query)

Found 1 matching videos for the query: 'Old Town Road'


In [26]:
print(len(does_song_exist_df))
display(does_song_exist_df.head(1))

1


,Song_Name,Video_ID
0,Lil Nas X - Old Town Road (Music Video) ft. Bi...,gUcisIlT7sM
